# 03 · Eksperimen training, hari 8 sampai 10

Baseline `v1_baseline_640` sudah ada dan aplikasinya sudah hidup. Notebook ini
mengejar angka, satu perubahan per run, dan **melaporkan yang gagal juga**.
Eksperimen yang gagal beserta hipotesis kenapa gagal bernilai lebih tinggi
daripada tabel yang semua barisnya naik.

## Rencananya sudah direvisi dua kali, dan alasannya dicatat

**Revisi pertama, setelah baseline.** Rencana semula menaruh "naikkan epoch"
sebagai eksperimen pertama. Itu terbantah oleh baselinenya sendiri. Early
stopping menyala di epoch 27 dengan hasil terbaik di epoch 17, artinya model
sudah berhenti membaik jauh sebelum batas 30 epoch. Menaikkan epoch saja hampir
pasti tidak menolong, jadi urutannya diganti dan `patience` dinaikkan dari 10
ke 20.

**Revisi kedua, setelah membaca kode Ultralytics.** Rencana semula memakai
`copy_paste` untuk kelas `no-helmet`. Itu **tidak akan bekerja di dataset
ini**. Blok `CopyPaste` di `ultralytics/data/augment.py` dibuka dengan

```python
if len(labels["instances"].segments) == 0 or self.p == 0:
    return labels
```

Tanpa segmentasi, ia keluar tanpa berbuat apa pun. Seluruh 7.724 baris label
dataset ini berisi lima kolom, yaitu bbox saja, tanpa poligon. Run itu akan
menghabiskan waktu GPU lalu menghasilkan model yang identik dengan run
sebelumnya, sementara namanya menjanjikan hal lain.

Penggantinya **oversampling tingkat data**, yaitu menduplikasi gambar yang
memuat `no-helmet`. Itu bukan augmentasi sama sekali, jadi ia tidak menyentuh
larangan augmentasi geometris di SOAL, dan ia menyerang persis angka 94
instance latih itu.

## Empat run

| Run | Yang diubah | Dibandingkan terhadap | Kenapa |
|---|---|---|---|
| `v2_imgsz960` | `imgsz` 640 ke 960 | v1 | EDA menunjukkan 32,6 persen helmet dan 47,9 persen no-helmet di bawah 32 piksel pada 640 |
| `v3_oversample_nohelmet` | train split, gambar no-helmet diduplikasi 4x | v2 | recall no-helmet 0,333 dari 24 instance, kelas terlemah |
| `v4_varian_s` | `yolo11n` ke `yolo11s` | v2 atau v3, yang lebih baik | terakhir, karena paling mahal dan paling jarang jadi akar masalah |
| `v5_dengan_mosaic` | mosaic dinyalakan | v2 | **pembanding, bukan kandidat.** Mengukur berapa mAP yang dikorbankan demi patuh pada SOAL |

## Yang diukur, dan kenapa bukan cuma mAP

Setiap run dinilai dua kali. Pertama mAP di test set dengan `conf=0.001`.
Kedua, **vonis per pekerja** dibandingkan vonis dari ground truth, memakai
`src/analitik.py` yang sama dengan yang dipakai aplikasi.

Yang kedua itu yang menentukan pemilihan. mAP mengukur kualitas kotak,
sementara yang dibaca pengawas adalah vonis, dan biaya ketiga jenis kesalahan
vonis sangat berbeda. Aturan pemilihannya ada di `skrip/eksperimen.py`,
dinyatakan di muka dan diuji di `tests/test_eksperimen.py`.

## Perkiraan waktu

Baseline 30 epoch pada 640 memakan 10,6 menit di T4. Pada 960 dengan patience
20, satu run bisa 40 sampai 90 menit. Empat run berarti beberapa jam, dan sesi
Colab gratis bisa mati di tengah jalan.

Karena itu **setiap run menyimpan catatannya ke Drive dan akan dilewati kalau
catatannya sudah ada**. Jalankan satu sel run per sesi kalau perlu, jangan
paksakan semuanya sekaligus.


In [3]:
!nvidia-smi -L || echo "TIDAK ADA GPU. Runtime, Change runtime type, T4 GPU."


GPU 0: Tesla T4 (UUID: GPU-54bd32ad-6859-fdbb-f887-0dab4c42924c)


In [4]:
# Versi dipin sama persis dengan requirements.txt aplikasi. Berkas bobot
# menyimpan referensi ke kelas Python di dalam library, jadi bobot yang dibuat
# versi lain berisiko gagal dimuat aplikasi.
!pip install -q "ultralytics==8.4.138"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 2.6 MB/s eta 0:00:00


In [5]:
import csv, glob, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

import torch, ultralytics, yaml
from ultralytics import YOLO

print("ultralytics", ultralytics.__version__)
print("torch      ", torch.__version__)
print("GPU        ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA")

assert ultralytics.__version__ == "8.4.138", (
    "Versi ultralytics berbeda dari yang dipin di requirements.txt aplikasi. "
    "Bobot yang dihasilkan berisiko gagal dimuat. Perbaiki dulu."
)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
ultralytics 8.4.138
torch       2.11.0+cu128
GPU         Tesla T4


## Menyiapkan dataset


In [6]:
ZIP_DRIVE = "/content/drive/MyDrive/capstone4/construction safety.v1i.yolov12.zip"
ROOT = "/content/dataset"

if not os.path.isdir(ROOT):
    if not os.path.exists(ZIP_DRIVE):
        from google.colab import drive
        drive.mount("/content/drive")
    assert os.path.exists(ZIP_DRIVE), f"Zip tidak ada di {ZIP_DRIVE}"
    shutil.copy(ZIP_DRIVE, "/content/data.zip")        # salin dulu, jangan baca dari mount
    with zipfile.ZipFile("/content/data.zip") as z:
        z.extractall(ROOT)

YAML = glob.glob(f"{ROOT}/*.yaml")[0]
cfg = yaml.safe_load(open(YAML))

# Roboflow menulis path relatif seperti ../train/images. Ditulis ulang menjadi
# absolut supaya tidak bergantung pada direktori kerja saat training.
for k, sub in [("train", "train"), ("val", "valid"), ("test", "test")]:
    cfg[k] = f"{ROOT}/{sub}/images"
with open(YAML, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

for s in ["train", "valid", "test"]:
    print(f"  {s:6} {len(glob.glob(f'{ROOT}/{s}/images/*')):5d} gambar")


Mounted at /content/drive
  train    997 gambar
  valid    119 gambar
  test      90 gambar


## Mengambil kode analisis dari repo

Lapisan analisis tidak ditulis ulang di sini. Notebook ini mengkloning repo dan
mengimpor `src/analitik.py` yang sama dengan yang dipakai aplikasi, supaya
angka vonis yang diukur di sini benar-benar angka yang akan dilihat pengguna.

Kalau kode analisis disalin ke dalam notebook, ia akan menyimpang dari
aplikasinya cepat atau lambat, dan angka di laporan menjadi angka dari sistem
yang berbeda.


In [7]:
REPO_URL = "https://github.com/Auliakukuhs/capstone4-apd-konstruksi"
REPO = "/content/repo"

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)

sys.path.insert(0, REPO)

from skrip.eksperimen import bangun_oversample, pilih_model, tabel_perbandingan
from skrip.validasi_analitik import bagian_dua

print("kode analisis terimpor dari", REPO)
print("commit          ", subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"],
                                         capture_output=True, text=True).stdout.strip())


kode analisis terimpor dari /content/repo
commit           30251f8


## Konfigurasi bersama dan fungsi penjalan

Satu fungsi menjalankan seluruh tahapan satu run, supaya tidak ada langkah yang
terlupa di satu run lalu dilakukan di run lain. Urutannya begini.

1. Lewati kalau catatannya sudah ada di Drive, karena sesi Colab bisa mati
2. Latih, dengan penurunan `batch` otomatis kalau memori GPU habis
3. **Verifikasi `args.yaml`**, bukan percaya kode yang baru ditulis
4. Evaluasi mAP di test set pada `conf=0.001`
5. Nilai vonis per pekerja lewat `src/analitik.py`
6. Simpan bobot dan catatan ke Drive

Langkah ketiga itu penting. Ultralytics menyalakan empat augmentasi geometris
secara default, dan satu salah ketik pada nama parameter akan diterima diam-diam
tanpa error.


In [8]:
PROJECT = "/content/runs_capstone"
DRIVE = "/content/drive/MyDrive/capstone4"
EPOCHS = 100
PATIENCE = 20        # dinaikkan dari 10, karena baseline berhenti terlalu cepat
BATCH = 16
SEED = 0
LEWATI_YANG_SUDAH = True

# Augmentasi geometris, seluruhnya nol mengikuti anjuran SOAL. mosaic dipisah
# karena satu run sengaja menyalakannya sebagai pembanding.
GEO_NOL = dict(
    fliplr=0.0, flipud=0.0, degrees=0.0, shear=0.0,
    perspective=0.0, translate=0.0, scale=0.0,
    mixup=0.0, cutmix=0.0, copy_paste=0.0,
)
# Fotometrik dipertahankan, sesuai anjuran SOAL.
FOTO = dict(hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, erasing=0.4)

os.makedirs(DRIVE, exist_ok=True)


def _epoch_info(dir_run):
    """Epoch yang benar-benar berjalan dan epoch terbaik, dibaca dari results.csv."""
    p = Path(dir_run) / "results.csv"
    if not p.exists():
        return None, None
    with open(p) as f:
        baris = [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(f)]
    if not baris:
        return None, None
    kolom = next((k for k in baris[0] if "mAP50-95" in k), None)
    terbaik = None
    if kolom:
        terbaik = 1 + max(range(len(baris)), key=lambda i: float(baris[i][kolom] or 0))
    return len(baris), terbaik


def jalankan_run(nama, *, model_awal="yolo11n.pt", yaml_data=None, imgsz=960,
                 batch=BATCH, epochs=EPOCHS, patience=PATIENCE, mosaic=0.0,
                 kandidat=True, tambahan=None):
    """Latih, verifikasi, evaluasi, nilai vonisnya, lalu catat."""
    jalur_catatan = f"{DRIVE}/{nama}_catatan.json"
    if LEWATI_YANG_SUDAH and os.path.exists(jalur_catatan):
        print(f"[{nama}] catatan sudah ada di Drive, dilewati.")
        return json.load(open(jalur_catatan, encoding="utf-8"))

    yaml_data = yaml_data or YAML
    aug = dict(GEO_NOL, mosaic=mosaic, **FOTO)

    t0 = time.time()
    batch_dipakai = batch
    while True:
        try:
            YOLO(model_awal).train(
                data=yaml_data, epochs=epochs, patience=patience,
                imgsz=imgsz, batch=batch_dipakai,
                seed=SEED, deterministic=True,
                project=PROJECT, name=nama, exist_ok=True,
                val=True, plots=True, **aug,
            )
            break
        except torch.cuda.OutOfMemoryError:
            if batch_dipakai <= 4:
                raise
            torch.cuda.empty_cache()
            batch_dipakai //= 2
            print(f"[{nama}] memori GPU habis, batch diturunkan ke {batch_dipakai}")
    menit = (time.time() - t0) / 60

    # Verifikasi args.yaml. Jangan percaya kode, percaya yang benar-benar dipakai.
    dir_run = f"{PROJECT}/{nama}"
    args = yaml.safe_load(open(f"{dir_run}/args.yaml"))
    salah = [k for k in GEO_NOL if args.get(k) not in (0, 0.0)]
    assert not salah, f"augmentasi geometris masih aktif: {salah}"
    assert args.get("mosaic") == mosaic, f"mosaic {args.get('mosaic')}, diminta {mosaic}"
    assert args.get("imgsz") == imgsz, f"imgsz {args.get('imgsz')}, diminta {imgsz}"
    print(f"[{nama}] verifikasi args.yaml lolos, mosaic {args.get('mosaic')}")

    best = f"{dir_run}/weights/best.pt"
    metrik = YOLO(best).val(data=yaml_data, split="test", imgsz=imgsz,
                            conf=0.001, iou=0.7, plots=False, verbose=False)
    per_kelas = {
        n: {"P": round(float(metrik.box.p[i]), 4), "R": round(float(metrik.box.r[i]), 4),
            "mAP50": round(float(metrik.box.ap50[i]), 4),
            "mAP50_95": round(float(metrik.box.ap[i]), 4)}
        for i, n in enumerate(metrik.names.values())
    }

    # Penilaian vonis per pekerja, memakai lapisan analisis aplikasi.
    # conf 0.25 karena itu nilai bawaan yang dilihat pengguna, bukan 0.001.
    label_test = sorted(Path(ROOT, "test", "labels").glob("*.txt"))
    vonis = bagian_dua(label_test, best, imgsz=imgsz, conf=0.25, iou=0.7)

    berjalan, terbaik = _epoch_info(dir_run)
    catatan = {
        "run": nama,
        "kandidat": kandidat,
        "model_awal": model_awal,
        "imgsz": imgsz,
        "epochs": epochs,
        "epochs_berjalan": berjalan,
        "epoch_terbaik": terbaik,
        "patience": patience,
        "batch": batch_dipakai,
        "batch_diminta": batch,
        "seed": SEED,
        "waktu_training_menit": round(menit, 1),
        "berhenti": "early stopping" if berjalan and berjalan < epochs else "epoch penuh",
        "ultralytics": ultralytics.__version__,
        "torch": torch.__version__,
        "python": sys.version.split()[0],
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "ukuran_bobot_mb": round(os.path.getsize(best) / 1e6, 2),
        "split_dilaporkan": "test",
        "conf_evaluasi": 0.001,
        "map50": round(float(metrik.box.map50), 4),
        "map50_95": round(float(metrik.box.map), 4),
        "per_kelas": per_kelas,
        "kecepatan_ms": {k: round(float(v), 1) for k, v in metrik.speed.items()},
        "augmentasi_geometris_mati": mosaic == 0.0,
        "mosaic": mosaic,
        "vonis_test": vonis,
    }
    if tambahan:
        catatan.update(tambahan)

    shutil.copy(best, f"{DRIVE}/apd_{nama}.pt")
    with open(jalur_catatan, "w", encoding="utf-8") as f:
        json.dump(catatan, f, ensure_ascii=False, indent=2)

    print(f"\n[{nama}] {menit:.1f} menit, batch {batch_dipakai}, "
          f"epoch {berjalan} terbaik {terbaik}")
    print(f"  mAP50 {catatan['map50']}   mAP50-95 {catatan['map50_95']}   "
          f"R no-helmet {per_kelas.get('no-helmet', {}).get('R')}")
    print(f"  akurasi vonis {vonis.get('akurasi_vonis')}   "
          f"pembebasan keliru {vonis.get('laju_pembebasan_keliru')}   "
          f"cakupan {vonis.get('cakupan_pekerja')}")
    return catatan


## v2, resolusi 960

Satu perubahan saja, `imgsz` 640 menjadi 960. Ini yang paling ditopang bukti.
EDA mengukur bahwa pada 640, sebanyak 32,6 persen kotak `helmet` dan 47,9
persen `no-helmet` jatuh di bawah 32 piksel setelah letterbox, yaitu definisi
objek small pada metrik COCO. Pada 960 angkanya turun menjadi 14,1 dan 17,0
persen.

Kalau `batch` terpaksa diturunkan karena memori habis, itu tercatat di `batch`
dan `batch_diminta`. Perbandingan terhadap v1 lalu membawa dua perubahan, bukan
satu, dan itu harus disebut saat melaporkan.


In [ ]:
v2 = jalankan_run("v2_imgsz960", imgsz=960)


New https://pypi.org/project/ultralytics/8.4.144 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, mo

## v3, oversampling kelas `no-helmet`

Dibandingkan terhadap v2, bukan terhadap v1, karena ia mewarisi `imgsz` 960.

`no-helmet` hadir di 49 dari 997 gambar train. Menduplikasi 49 gambar itu empat
kali menaikkan porsinya dari 1,5 menjadi 4,7 persen sementara kelas lain hampir
tidak bergerak, dan gambar train hanya bertambah 14,7 persen. Angka itu sudah
diverifikasi di komputer lokal sebelum notebook ini dijalankan, jadi sel
berikut seharusnya mencetak angka yang sama.

**Risikonya dinyatakan terbuka.** Yang diduplikasi hanya 49 foto berbeda, dan
model melihat masing-masing empat kali per epoch. Kalau `no-helmet` membaik
tapi kelas lain memburuk, overfitting pada 49 foto itu tersangka pertamanya.


In [ ]:
ROOT_OS = "/content/dataset_oversample"

stat_os = bangun_oversample(Path(ROOT, "train"), Path(ROOT_OS, "train"), kali=4)
print(json.dumps(stat_os, indent=2, ensure_ascii=False))

# valid dan test TIDAK disentuh. Oversampling di sana akan membuat metriknya
# tidak sebanding dengan run lain, dan itu bentuk kebocoran yang paling halus.
for sub in ("valid", "test"):
    tujuan = Path(ROOT_OS, sub)
    if not tujuan.exists():
        shutil.copytree(Path(ROOT, sub), tujuan)

YAML_OS = f"{ROOT_OS}/data.yaml"
cfg_os = dict(yaml.safe_load(open(YAML)))
cfg_os["train"] = f"{ROOT_OS}/train/images"
cfg_os["val"] = f"{ROOT_OS}/valid/images"
cfg_os["test"] = f"{ROOT_OS}/test/images"
with open(YAML_OS, "w") as f:
    yaml.safe_dump(cfg_os, f, sort_keys=False, allow_unicode=True)

print()
for s in ("train", "valid", "test"):
    print(f"  {s:6} {len(glob.glob(f'{ROOT_OS}/{s}/images/*')):5d} gambar")
assert stat_os["porsi_target_sesudah"] > stat_os["porsi_target_sebelum"] * 2.5


{
  "kelas_target": "no-helmet",
  "kali": 4,
  "gambar_asal": 997,
  "gambar_memuat_target": 49,
  "gambar_hasil": 1144,
  "instance_sebelum": {
    "helmet": 2116,
    "no-helmet": 94,
    "no-vest": 741,
    "person": 2362,
    "vest": 1073
  },
  "instance_sesudah": {
    "helmet": 2425,
    "no-helmet": 376,
    "no-vest": 1029,
    "person": 2935,
    "vest": 1286
  },
  "porsi_target_sebelum": 0.0147,
  "porsi_target_sesudah": 0.0467
}

  train   1144 gambar
  valid    119 gambar
  test      90 gambar


In [ ]:
v3 = jalankan_run("v3_oversample_nohelmet", imgsz=960, yaml_data=YAML_OS,
                  tambahan={"oversample": stat_os})


New https://pypi.org/project/ultralytics/8.4.144 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_oversample/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mod

## v4, varian model `s`

Dijalankan terakhir di antara kandidat, karena paling mahal dan paling jarang
menjadi akar masalah. Bobot `yolo11s` sekitar 19 MB, masih di bawah batas 50 MB
GitHub dan masih muat di memori Streamlit Community Cloud yang sekitar 1 GB,
tapi inference-nya lebih lambat, dan itu terasa saat aplikasi dipakai.

Basisnya konfigurasi yang lebih baik antara v2 dan v3. Sel berikut memilihnya
sendiri dari angka vonis, bukan dari mAP, memakai aturan yang sama dengan
pemilihan akhir.


In [ ]:
# Basis v4 dipilih dari dua run sebelumnya, memakai aturan yang sama.
basis, alasan_basis, _ = pilih_model([v2, v3])
print(alasan_basis)

pakai_os = basis == "v3_oversample_nohelmet"
v4 = jalankan_run("v4_varian_s", model_awal="yolo11s.pt", imgsz=960,
                  yaml_data=YAML_OS if pakai_os else YAML,
                  tambahan={"basis": basis, "memakai_oversample": pakai_os})


Terpilih v3_oversample_nohelmet. Laju pembebasan keliru 0.0357, terendah di antara 2 run yang lolos penjaga cakupan. Akurasi vonis 0.6667, cakupan pekerja 0.8411, mAP@0.5:0.95 0.3342.
New https://pypi.org/project/ultralytics/8.4.144 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_oversample/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript,

## v5, pembanding dengan mosaic dinyalakan

**Run ini bukan kandidat.** Ia sengaja melanggar batasan SOAL, dan tidak akan
dipakai aplikasi apa pun hasilnya.

Gunanya menjawab satu pertanyaan yang pantas diajukan. Mematikan
mosaic biasanya menurunkan mAP pada dataset kecil. Berapa besar penurunan itu
di dataset ini?

Tanpa run ini, pernyataan "sedikit mAP dikorbankan demi kepatuhan" hanya klaim.
Dengan run ini, ia menjadi angka.

`skrip/eksperimen.py` mengeluarkan run bertanda `kandidat=False` dari pemilihan,
dan itu diuji di `tests/test_eksperimen.py`, jadi ia tidak mungkin ikut terpilih
karena kelalaian.


In [9]:
v5 = jalankan_run("v5_dengan_mosaic", imgsz=960, mosaic=1.0, kandidat=False)


[v5_dengan_mosaic] catatan sudah ada di Drive, dilewati.


## Perbandingan seluruh run

v1 dibaca dari repo, sisanya dari Drive. Kalau ada run yang belum dijalankan, ia
tidak muncul di tabel, dan itu memang yang diinginkan. Lebih baik barisnya tidak
ada daripada berisi angka kosong yang terbaca sebagai nol.


In [10]:
semua = []

# v1 dari repo, karena catatannya sudah masuk repo sejak hari 3
v1_jalur = Path(REPO, "laporan", "v1_baseline_640_catatan.json")
if v1_jalur.exists():
    v1 = json.load(open(v1_jalur, encoding="utf-8"))
    v1.setdefault("kandidat", True)
    semua.append(v1)

for nama in ("v2_imgsz960", "v3_oversample_nohelmet", "v4_varian_s", "v5_dengan_mosaic"):
    p = Path(DRIVE, f"{nama}_catatan.json")
    if p.exists():
        semua.append(json.load(open(p, encoding="utf-8")))

baris = tabel_perbandingan(semua)
kolom = ["run", "imgsz", "model_awal", "epoch_terbaik", "map50", "map50_95",
         "R_no_helmet", "akurasi_vonis", "laju_pembebasan_keliru",
         "laju_tuduhan_palsu", "cakupan_pekerja", "kandidat"]

lebar = {k: max([len(k)] + [len(str(b.get(k))) for b in baris]) for k in kolom}
print("  ".join(k.ljust(lebar[k]) for k in kolom))
print("-" * (sum(lebar.values()) + 2 * (len(kolom) - 1)))
for b in baris:
    print("  ".join(str(b.get(k)).ljust(lebar[k]) for k in kolom))

# v1 dievaluasi pada 640 dan catatannya belum punya angka vonis, jadi kolom
# vonisnya kosong. Untuk membuatnya sebanding, jalankan
# skrip/validasi_analitik.py pada bobot v1 di komputer sendiri.


run                     imgsz  model_awal  epoch_terbaik  map50   map50_95  R_no_helmet  akurasi_vonis  laju_pembebasan_keliru  laju_tuduhan_palsu  cakupan_pekerja  kandidat
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------
v1_baseline_640         640    yolo11n.pt  17             0.7224  0.3704    0.3333       0.6952         0.0702                  0.0673              0.8738           True    
v2_imgsz960             960    yolo11n.pt  16             0.6909  0.3359    0.4583       0.6776         0.0536                  0.06                0.8551           True    
v3_oversample_nohelmet  960    yolo11n.pt  49             0.6843  0.3342    0.4583       0.6667         0.0357                  0.0198              0.8411           True    
v4_varian_s             960    yolo11s.pt  30             0.6503  0.3187    0.3333       0.6742         0.0556                  0.

## Memilih model, dengan aturan yang dinyatakan di muka

Aturannya ada di `skrip/eksperimen.py`, berurutan.

1. Run bertanda bukan kandidat dikeluarkan
2. **Penjaga cakupan.** Run yang cakupan pekerjanya jatuh lebih dari 5 poin di
   bawah yang terbaik dikeluarkan. Tanpa penjaga ini, model yang hampir tidak
   mendeteksi siapa pun akan terlihat unggul, sebab laju kesalahan dihitung
   dari pekerja yang berhasil dipasangkan dan penyebutnya menyusut
3. **Laju pembebasan keliru terendah menang.** Pelanggar yang dinyatakan
   lengkap adalah kesalahan termahal, karena ia menghentikan pemeriksaan
   terhadap orang yang justru berisiko
4. Seri diputus akurasi vonis, lalu mAP@0.5:0.95

mAP ada di urutan terakhir, dan itu disengaja. Ia mengukur kualitas kotak,
sementara yang dipakai pengawas adalah vonis.


In [11]:
terpilih, alasan, terurut = pilih_model(semua)
print(alasan)
print()

if terpilih:
    bobot = Path(DRIVE, f"apd_{terpilih}.pt")
    if bobot.exists():
        print(f"bobot terpilih  {bobot}  ({bobot.stat().st_size / 1e6:.2f} MB)")
    else:
        print(f"bobot {bobot} tidak ada di Drive")

# Tabel disimpan supaya README dan notebook mengutip sumber yang sama.
ringkas = {"terpilih": terpilih, "alasan": alasan, "run": baris}
with open(f"{DRIVE}/perbandingan_run.json", "w", encoding="utf-8") as f:
    json.dump(ringkas, f, ensure_ascii=False, indent=2)
print(f"\ntabel -> {DRIVE}/perbandingan_run.json")


Terpilih v3_oversample_nohelmet. Laju pembebasan keliru 0.0357, terendah di antara 4 run yang lolos penjaga cakupan. Akurasi vonis 0.6667, cakupan pekerja 0.8411, mAP@0.5:0.95 0.3342.

bobot terpilih  /content/drive/MyDrive/capstone4/apd_v3_oversample_nohelmet.pt  (5.54 MB)

tabel -> /content/drive/MyDrive/capstone4/perbandingan_run.json


## Alur setelah notebook ini

Bobot terpilih beserta catatan versinya dan `perbandingan_run.json` diunduh
dari Drive. Bobotnya masuk ke `models/`, kedua json ke `laporan/`.

Resolusi bawaan aplikasi mengikuti catatan versi bobot yang dipilih, jadi bobot
640 dan 960 bisa berdampingan di repo yang sama tanpa mengubah kode.

Seluruh uji dijalankan ulang setelah bobot berganti, dan angka di bagian 4, 5,
dan 8 README diambil dari tabel di atas.

## Yang sengaja tidak dikerjakan di notebook ini

**Tuning banyak hyperparameter sekaligus.** Satu perubahan per run adalah harga
yang dibayar supaya penyebab perbaikan dapat ditunjuk dengan pasti.

**Menyembunyikan run yang gagal.** Kalau v3 atau v4 tidak memperbaiki apa pun,
barisnya tetap masuk tabel beserta hipotesis kenapa gagal. Tabel yang semua
barisnya naik hampir selalu berarti ada yang tidak dilaporkan.

**Evaluasi final, confusion matrix, dan kurva PR.** Itu hari 11, di notebook
terpisah, pada satu model terpilih saja.
